In [ ]:
import importlib.util, subprocess, sys

NEEDED = {
    "dotenv": "python-dotenv",
    "langchain_openai": "langchain-openai",
    "langchain_community": "langchain-community>=0.3.0,<0.4.0",
    "langchain_text_splitters": "langchain-text-splitters",
    "faiss": "faiss-cpu>=1.7.4",
    "pypdf": "pypdf>=4.0.0",
}
missing = [req for mod, req in NEEDED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("All required libraries are installed.")

In [ ]:
GATEWAY = "https://openai-api-management-gw.azure-api.net"

//2ABecnfxzhRg4M5D6pBKiqxXVhmGB2WvQ0aYKkbTCPsj0JLKsZPfJQQJ99BDAC77bzfXJ3w3AAABACOGi3sC

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=True)
key = ""
assert key, "KEY not found. Add it to the .env file in this notebook's folder."
print("API key loaded.")

In [ ]:
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings

GATEWAY = "https://openai-api-management-gw.azure-api.net"

llm = AzureChatOpenAI(
    base_url=f"{GATEWAY}/openaiprodtest-clone",
    model="gpt-5-mini",
    api_version="2024-12-01-preview",
    api_key=key,
    validate_base_url=False,
)

embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=GATEWAY,
    azure_deployment="text-embedding-ada-002",
    api_version="2023-05-15",
    api_key=key,
)

print("Chat model replied:", llm.invoke("Reply with just the word: ready").content)
print("Embeddings OK, vector length:", len(embeddings.embed_query("test")))

In [ ]:
question = "How many days of paid annual leave does claude Corporation offer its employees?"

no_rag_answer = llm.invoke(question).content
print("--- LLM ANSWER WITHOUT RAG ---\n")
print(no_rag_answer)

In [ ]:
from langchain_community.document_loaders import TextLoader

docs = TextLoader("hr_policy.txt").load()     # the whole file becomes one Document

print(f"Loaded hr_policy.txt: {len(docs[0].page_content)} characters\n")
print(docs[0].page_content[:600])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
hr_chunks = splitter.split_documents(docs)

print(f"Split the document into {len(hr_chunks)} chunks.\n")
print("--- Example chunk ---")
print(hr_chunks[3].page_content)

In [ ]:
INDEX_DIR = "hr_policy_txt_faiss_index"
chunk_texts = [chunk.page_content for chunk in hr_chunks]
chunk_vectors = embeddings.embed_documents(chunk_texts)      # one API call for all chunks

print(f"Created {len(chunk_vectors)} vectors, one per chunk.")
print(f"Each vector has {len(chunk_vectors[0])} numbers.")

In [ ]:
chunk_vectors[0]

In [ ]:
from langchain_community.vectorstores import FAISS

hr_vectorstore = FAISS.from_embeddings(
    text_embeddings=list(zip(chunk_texts, chunk_vectors)),   # (text, vector) pairs we already computed
    embedding=embeddings,                                     # used later to turn QUESTIONS into vectors
    metadatas=[chunk.metadata for chunk in hr_chunks],
)
print("Vector database created.")

# price<=500 & season=="summer"


In [ ]:
print(f"Vectors stored:       {hr_vectorstore.index.ntotal}")
print(f"Numbers per vector:   {hr_vectorstore.index.d}")

first_id = hr_vectorstore.index_to_docstore_id[0]           # each vector position points to a stored chunk
first_doc = hr_vectorstore.docstore.search(first_id)
print(f"\nVector #0 is linked to this chunk:\n{first_doc.page_content[:200]} ...")

In [ ]:
INDEX_DIR = "hr_policy_txt_faiss_index"

hr_vectorstore.save_local(INDEX_DIR)
for f in sorted(os.listdir(INDEX_DIR)):
    print(f"Saved {INDEX_DIR}/{f}  ({os.path.getsize(os.path.join(INDEX_DIR, f)):,} bytes)")

# Later (or in another notebook), load it instead of re-embedding everything:
hr_vectorstore = FAISS.load_local(INDEX_DIR, embeddings, allow_dangerous_deserialization=True)
print(f"\nLoaded back from disk: {hr_vectorstore.index.ntotal} vectors.")

In [ ]:
question="How many days of paid annual leave does ACME Corporation offer its employees?"

In [ ]:
results = hr_vectorstore.similarity_search_with_score(question, k=2)

print(f"Question: {question}\n")
for doc, score in results:
    print(f"Distance {score:.4f}  (smaller = closer in meaning)")
    print(f"  {doc.page_content[:120]} ...\n")

In [ ]:
PROMPT_TEMPLATE = """Answer the question using ONLY the context below. If the answer isn't in the
context, say you don't have that information -- do not guess.

Context:
{context}

Question: {question}

Answer:"""


In [ ]:
def ask_hr_policy(q):
    chunks = hr_vectorstore.similarity_search(q, k=3)                      # retrieve
    prompt = PROMPT_TEMPLATE.format(
        context="\n\n".join(c.page_content for c in chunks), question=q)   # augment
    print(prompt)
    return llm.invoke(prompt).content                                      # generate

for q in [
    "How many days of paid annual leave does ACME Corporation offer its employees?",
    "How many days of paternity leave are employees entitled to?",
    "Can sick leave be carried forward to next year?",
    "What is ACME's policy on employee stock options?",   # deliberately NOT in the file
]:
    print(f"Q: {q}\nA: {ask_hr_policy(q)}\n")